:::{admonition} Download
:class: important

Download this notebook: **{nb-download}`population.ipynb`**!

:::

In [1]:
%matplotlib inline
import warnings

# Ignore the first specific warning
warnings.filterwarnings(
    "ignore",
    message="plotting functions contained within `_documentation_utils` are intended for nemos's documentation.",
    category=UserWarning,
)

# Ignore the second specific warning
warnings.filterwarnings(
    "ignore",
    message="Ignoring cached namespace 'core'",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message=(
        "invalid value encountered in div "
    ),
    category=RuntimeWarning,
)

(glm-population)=
# Population GLM

[`PopulationGLM`](nemos.glm.PopulationGLM) fits a GLM to every neuron of a population at once. It takes the same arguments and has the same methods as the [single neuron GLM](glm-single-neuron). This page covers only what changes:

- `y` has one column per neuron,
- every parameter gains a trailing neuron axis,
- each neuron can have its own set of predictors.

Fitting the population jointly is faster than fitting one neuron at a time in a loop. JAX vectorizes the fit across neurons and can run it on a GPU. The loop uses less memory.

:::{note}

For an unregularized, Lasso, or Ridge GLM, the loss is a sum of one term per neuron. Fitting the population jointly and fitting each neuron on its own give the same parameters. The same holds for a GroupLasso whose groups do not mix neurons, so that the parameters of a neuron fall in groups of their own, which is what the default mask gives you.

:::

## Initializing a PopulationGLM object

The arguments are those of [`GLM`](nemos.glm.GLM), with the same defaults, plus a [`feature_mask`](neuron-specific-features) that selects the predictors of each neuron.

In [2]:
import nemos as nmo

nmo.glm.PopulationGLM()

,observation_model,PoissonObservations()
,inverse_link_function,<function exp...x7fbc96ffe660>
,regularizer,UnRegularized()
,fix_params,"(None, ...)"
,solver_name,'LBFGS'
,solver_kwargs,{}
,regularizer_strength,None
,fit_intercept,True
,feature_mask,None
Name,Type,Value
aux_,NoneType,None


(glm-population-methods)=
## Methods

The inputs are the same as for the [single neuron GLM](glm-input-types). `y` is now two-dimensional, with shape `(n_samples, n_neurons)`.

### Fit

In [3]:
# X of shape (n_samples, n_features), y of shape (n_samples, n_neurons)
X, y = nmo.datasets.tuned_poisson(n_samples=1000, n_neurons=2)

model = nmo.glm.PopulationGLM().fit(X, y)
model

,observation_model,PoissonObservations()
,inverse_link_function,<function exp...x7fbc96ffe660>
,regularizer,UnRegularized()
,fix_params,"(None, ...)"
,solver_name,'LBFGS'
,solver_kwargs,{}
,regularizer_strength,None
,fit_intercept,True
,feature_mask,None
Name,Type,Value
aux_,NoneType,None


#### Pynapple Inputs

`y` can be a pynapple `TsdFrame`, one column per neuron. `fit` stores its column names and metadata. [`predict`](nemos.glm.PopulationGLM.predict) and [`simulate`](nemos.glm.PopulationGLM.simulate) attach them to their outputs.

In [4]:
import numpy as np
import pynapple as nap

# suppress jax to numpy conversion warning
nap.nap_config.suppress_conversion_warnings = True

# 10 ms bins
time = np.arange(X.shape[0]) / 100

predictors = nap.TsdFrame(t=time, d=np.asarray(X))
counts = nap.TsdFrame(
    t=time,
    d=np.asarray(y),
    columns=["unit0", "unit1"],
    metadata={"region": ["M1", "PFC"]},
)

model_nap = nmo.glm.PopulationGLM().fit(predictors, counts)

#### PyTree Inputs

With `X` a PyTree, each leaf of `coef_` has the shape of its predictors and a trailing neuron axis.

In [5]:
# the coupling filters play the role of the single neuron history
X_dict = {"stimulus": X[:, :4], "coupling": X[:, 4:]}

model_dict = nmo.glm.PopulationGLM().fit(X_dict, y)

{key: coef.shape for key, coef in model_dict.coef_.items()}

{'coupling': (4, 2), 'stimulus': (4, 2)}

#### Fitted Attributes

Every fitted attribute has one entry per neuron, along the last axis.

In [6]:
print(f"coef_:       {model.coef_.shape}")
print(f"intercept_:  {model.intercept_.shape}")
print(f"scale_:      {model.scale_}")
print(f"dof_resid_:  {model.dof_resid_}")

coef_:       (8, 2)
intercept_:  (2,)
scale_:      [1. 1.]
dof_resid_:  [991. 991.]


Column `j` of `coef_` holds the weights of neuron `j`. Entry `j` of `intercept_`, `scale_` and `dof_resid_` holds its baseline, dispersion and residual degrees of freedom. The single neuron [`GLM`](nemos.glm.GLM) stores the same quantities without the neuron axis.

#### NaN Treatment

`fit` drops a sample when `X` or any column of `y` contains a NaN. A NaN in a single neuron therefore drops that sample for the whole population.

In [7]:
y_nan = np.asarray(y, dtype=float).copy()
# a gap in the recording of the first neuron only
y_nan[:10, 0] = np.nan

model_nan = nmo.glm.PopulationGLM().fit(X, y_nan)
model_dropped = nmo.glm.PopulationGLM().fit(np.asarray(X)[10:], np.asarray(y)[10:])

np.allclose(model_nan.coef_, model_dropped.coef_)

True

To keep the samples of the other neurons, fit the neuron with the gap on its own using a [`GLM`](nemos.glm.GLM).

### Predict

[`predict`](nemos.glm.PopulationGLM.predict) returns one rate per sample and neuron, in counts per bin.

In [8]:
model.predict(X).shape

(1000, 2)

With pynapple predictors, `predict` returns a `TsdFrame`, one column per unit, where the single neuron GLM returns a `Tsd`.

In [9]:
rate = model_nap.predict(predictors)
rate

Time (s)    unit0                unit1
----------  -------------------  -------------------
0.0         5.155981063842773    0.03784782066941261
0.01        5.667296409606934    0.05499938130378723
0.02        3.1670897006988525   7.369950771331787
0.03        9.125699996948242    2.2817256450653076
0.04        9.851353645324707    0.663364052772522
0.05        8.616137504577637    0.2864370346069336
0.06        8.737861633300781    0.304229736328125
...
9.93        6.120498180389404    4.760712623596191
9.94        7.940459728240967    0.20547819137573242
9.95        5.640016555786133    5.137512683868408
9.96        0.8056718111038208   10.362226486206055
9.97        0.08909513801336288  7.548639297485352
9.98        9.97890853881836     0.8808091282844543
9.99        9.360177993774414    2.0353920459747314
Metadata
region      M1                   PFC
dtype: float32, shape: (1000, 2)

A `TsdFrame` can carry metadata labeling its columns, which here are the units of the population. `predict` carries over the metadata of the counts the model was fit on.

In [10]:
rate.metadata

,region
unit0,M1
unit1,PFC


The rate can then be split by any of those labels, without matching columns back to units by hand.

In [11]:
rate[:, rate.region == "M1"]

Time (s)    unit0
----------  -------------------
0.0         5.155981063842773
0.01        5.667296409606934
0.02        3.1670897006988525
0.03        9.125699996948242
0.04        9.851353645324707
0.05        8.616137504577637
0.06        8.737861633300781
...
9.93        6.120498180389404
9.94        7.940459728240967
9.95        5.640016555786133
9.96        0.8056718111038208
9.97        0.08909513801336288
9.98        9.97890853881836
9.99        9.360177993774414
Metadata
region      M1
dtype: float32, shape: (1000, 1)

The column names and the metadata are the only part of the output that comes from the fit. The type and the timestamps follow the predictors passed to `predict`.

### Score

[`score`](nemos.glm.PopulationGLM.score) returns a single number, the average over samples and neurons, which is what scikit-learn cross-validation expects from an estimator.

In [12]:
model.score(X, y, score_type="pseudo-r2-McFadden")

Array(0.5044401, dtype=float32)

`score` applies `aggregate_sample_scores` to an array of shape `(n_samples, n_neurons)`. Averaging over the sample axis alone leaves one score per neuron.

In [13]:
import jax.numpy as jnp

model.score(
    X,
    y,
    score_type="pseudo-r2-McFadden",
    aggregate_sample_scores=lambda x: jnp.mean(x, axis=0),
)

Array([0.5064955, 0.502414 ], dtype=float32)

### Simulate

[`simulate`](nemos.glm.PopulationGLM.simulate) draws counts for every neuron and returns them with the rate they were drawn from, both of shape `(n_samples, n_neurons)`.

In [14]:
import jax

counts_sim, rate_sim = model.simulate(jax.random.key(0), X)

print(f"counts: {counts_sim.shape}")
print(f"rate:   {rate_sim.shape}")

counts: (1000, 2)
rate:   (1000, 2)


The simulation is feed-forward: the counts drawn do not feed back into the rate through the coupling filters. For a recurrently coupled population, use [`simulate_recurrent`](nemos.simulation.simulate_recurrent), shown in [simulate a coupled population](../../../how_to_guide/simulate_coupled_population.md).

### Fitting out of memory

[`stochastic_fit`](nemos.glm.PopulationGLM.stochastic_fit) and [`update`](nemos.glm.PopulationGLM.update) take the same arguments and behave as for the [single neuron GLM](glm-out-of-memory). The loader serves batches of `y` with all the neurons in them.

(glm-population-model-configuration)=
## Model Configurations

You set observation models, inverse link functions, regularizers and solvers exactly as in the [single neuron GLM](glm-model-configuration).

A parameter-wise `regularizer_strength` is the one argument whose shape changes: it has the shape of the parameters, neuron axis included. The array of one strength per feature that the single neuron GLM takes is rejected here.

In [15]:
n_features, n_neurons = X.shape[1], y.shape[1]

# penalize the second neuron heavily, the first one barely
strength = np.full((n_features, n_neurons), 0.01)
strength[:, 1] = 100.0

nmo.glm.PopulationGLM(regularizer="Ridge", regularizer_strength=strength).fit(X, y).coef_

Array([[ 7.4665946e-01, -2.9338736e-03],
       [ 1.2706635e+00, -5.2223615e-03],
       [ 1.5138890e+00, -5.3804168e-03],
       [ 1.4782633e+00, -2.5246902e-03],
       [ 2.6855415e-01,  4.9726772e-03],
       [-1.3527025e+00,  6.8241996e-03],
       [-2.2507617e+00,  3.3665833e-03],
       [-1.6745650e+00,  8.9788315e-04]], dtype=float32)

The group-wise and parameter-wise forms, and the special cases of ElasticNet and GroupLasso, are covered in [regularizers](../../regularizers.md).

### GroupLasso

[`GroupLasso`](nemos.regularizer.GroupLasso) defines its groups per neuron. With the default mask, each leaf of `coef` is a separate group for each neuron. A group of predictors can then shrink to zero for one neuron and stay for another.

In [16]:
model_group = nmo.glm.PopulationGLM(
    regularizer="GroupLasso",
    regularizer_strength=0.3,
).fit(X_dict, y)

# norm of each group, one entry per neuron
{key: np.linalg.norm(coef, axis=0) for key, coef in model_group.coef_.items()}

{'coupling': array([0.      , 1.067085], dtype=float32),
 'stimulus': array([1.0269294, 0.       ], dtype=float32)}

With 2 leaves and 2 neurons there are 4 groups. Here, the first neuron keeps its stimulus weights and drops its coupling. The second neuron does the opposite.

A mask of your own has shape `(n_groups, n_features, n_neurons)` for each leaf, with the neuron axis like the parameters.

In [17]:
# group 0: the stimulus of every neuron; group 1: the coupling of every neuron
mask = np.zeros((2, n_features, n_neurons))
mask[0, :4] = 1
mask[1, 4:] = 1

nmo.glm.PopulationGLM(
    regularizer=nmo.regularizer.GroupLasso(mask=mask),
    regularizer_strength=0.3,
).fit(X, y)

,observation_model,PoissonObservations()
,inverse_link_function,<function exp...x7fbc96ffe660>
,regularizer,GroupLasso(ma...tercept=None))
,regularizer_strength,0.3
,fix_params,"(None, ...)"
,solver_name,'ProximalGradient'
,solver_kwargs,{}
,regularizer__mask,GLMParams(coe...ntercept=None)
,fit_intercept,True
,feature_mask,None
Name,Type,Value


:::{warning}

A group that spans several neurons, like the two above, couples their fits. The joint fit then differs from fitting each neuron on its own.

:::

(glm-population-parameter-specification)=
## Parameter specification

`fit_intercept` and `fix_params` work as in the [single neuron GLM](glm-parameter-specification). The values held fixed have the shape of the parameters, with the neuron axis last.

### Fix parameter values

In [18]:
# hold one intercept per neuron, learn the weights
model_fixed = nmo.glm.PopulationGLM(
    fix_params=(None, np.array([0.5, 1.0]))
).fit(X, y)

print(f"intercept_: {model_fixed.intercept_}")

intercept_: [0.5 1. ]


With `X` a PyTree, the model holds or learns each leaf of `coef` on its own, for all the neurons together.

In [19]:
# hold the coupling weights of every neuron at zero, learn the stimulus ones
fixed = ({"stimulus": None, "coupling": np.zeros((4, n_neurons))}, None)

model_fixed = nmo.glm.PopulationGLM(fix_params=fixed).fit(X_dict, y)

model_fixed.coef_["coupling"]

Array([[0., 0.],
       [0., 0.],
       [0., 0.],
       [0., 0.]], dtype=float32)

To remove a group of predictors from some neurons only, use a feature mask.

(neuron-specific-features)=
### Neuron-specific features

`feature_mask` selects which predictors enter the model of each neuron. It has the shape of `coef_`, with a 1 where the model learns the weight and a 0 where it holds the weight at zero.

Take three predictors and two neurons:

- predictor 0 drives both neurons,
- predictor 1 drives only the first neuron,
- predictor 2 drives only the second neuron.

In [20]:
input_features = X[:, :3]

feature_mask = np.ones((3, n_neurons))
# drop predictor 2 from the first neuron
feature_mask[2, 0] = 0
# drop predictor 1 from the second neuron
feature_mask[1, 1] = 0

model_masked = nmo.glm.PopulationGLM(feature_mask=feature_mask).fit(input_features, y)

model_masked.coef_

Array([[ -0.36547473, -11.535214  ],
       [  1.7087805 ,   0.        ],
       [  0.        ,  -4.8226867 ]], dtype=float32)

`coef_[i, j]` is zero wherever `feature_mask[i, j]` is zero. You can pass the mask at initialization or set it as an attribute before fitting. After fitting, the attribute still returns the mask, but assigning to it raises.

A masked weight is not estimated, so it does not count towards the residual degrees of freedom of its neuron.

In [21]:
unmasked = nmo.glm.PopulationGLM().fit(input_features, y)

print(f"masked:   {model_masked.dof_resid_}")
print(f"unmasked: {unmasked.dof_resid_}")

masked:   [997. 997.]
unmasked: [996. 996.]


#### Masking groups of features

With `X` a PyTree, the mask has the same structure, each leaf shaped like its coefficients.

In [22]:
# the stimulus drives both neurons, the coupling only the second one
feature_mask = {
    "stimulus": np.ones((4, n_neurons)),
    "coupling": np.array([[0, 1]] * 4),
}

model_dict = nmo.glm.PopulationGLM(feature_mask=feature_mask).fit(X_dict, y)

model_dict.coef_["coupling"]

Array([[0.       , 1.4873959],
       [0.       , 1.9396404],
       [0.       , 1.2631433],
       [0.       , 1.1648017]], dtype=float32)

A single column of zeros removes the entire `"coupling"` group from the predictors of the first neuron. You do not have to index the columns of the design one by one. Use this form when the mask encodes a hypothesis about which inputs reach which neuron. Examples are dropping self-coupling or restricting coupling to a sub-population.

## Examples

```{nemos-examples}
:model: PopulationGLM
```